# Assemble Stage 2 evaluation table after per-target SNR jobs finish


Note that you will need to start the kernel in the main working directory of the repo so imports and relative file paths work. If working in an IDE, this might mean configuring IDE settings.

In [1]:
import re
import os

import paths
os.chdir(paths.stage1_code)

import numpy as np
from tqdm import tqdm
from astropy import table
from astropy.table import Table, QTable
from astropy import units as u

import database_utilities as dbutils
import catalog_utilities as catutils

from processing import target_lists
from processing import preloads
from processing import transit_evaluation_utilities as tutils



In [2]:
# settings
sigma_threshold = 1

In [3]:
# targets
target_sets = [target_lists.eval_no(i) for i in range(1,4)]
targets = sum(target_sets, [])
drops = ['toi-4336a', 'hd21520', 'toi-6992', 'wasp-84', 'v1298tau'] # shouldn't need these in the future as I de-flagged them in the obsprog sheet
targets = set(targets) - set(drops)
targets = sorted(list(targets))

In [4]:
# paths
path_staging_area = paths.packages / '2026-03-10.stage2.eval3.staging_area'
path_planet_catalog = path_staging_area / 'planet_catalog_all_evals.ecsv'
path_host_catalog = path_staging_area / 'host_catalog_all_evals.ecsv'


In [5]:
##%% load catalogs

with catutils.catch_QTable_unit_warnings():
    planet_catalog = QTable.read(path_planet_catalog)
    host_catalog = QTable.read(path_host_catalog)

planet_catalog.add_index('tic_id')
host_catalog.add_index('tic_id')



In [6]:
def path_snrs(planet, host, tst_type):
    filenamer = tutils.FileNamer(tst_type, planet, host)
    return filenamer.snr_tbl_full

def load_snr_db(planet, host, tst_type):
    path = path_snrs(planet, host, tst_type)
    return tutils.DetectabilityDatabase.from_file(path)

def get_lya_flux(host):
    lya = host.lya_reconstruction
    Flya = np.trapz(lya.fluxes[0], lya.wavegrid_earth)
    return Flya

In [7]:
##%% QA settings

from datetime import datetime

N_CASE_DICT = {
    'eta': 4,
    'Tion': 4,
    'sw_ram_pressure_at_pl': 4,
    'sw_velocity': 3,
    'mass': 4,
    'time offset': 17,
    'lya reconstruction case': 5,
}
N_STIS_APERTURES = 4
N_COS_APERTURES = 1

STALE_CUTOFF = datetime(2026, 5, 1)  # user-adjustable each run
RUN_OFFSET_STATS_SMOKE_TEST = False  # True only when offset_stats smoke test is desired


In [8]:
##%% QA detection-sigmas tables

from processing.detection_sigmas_qa import (
    validate_all_targets,
    print_qa_report,
    qa_summary_table,
    planet_qa_reason,
    planet_cos_qa_reason,
    cos_metric_column_names,
    planet_key as snr_planet_key,
)

qa_issues, snr_excluded_planets, cos_incomplete_planets = validate_all_targets(
    sorted(targets),
    host_catalog,
    planet_catalog,
    stale_cutoff=STALE_CUTOFF,
    case_dict=N_CASE_DICT,
    n_stis_apertures=N_STIS_APERTURES,
    n_cos_apertures=N_COS_APERTURES,
    run_offset_stats_smoke_test=RUN_OFFSET_STATS_SMOKE_TEST,
    sigma_threshold=sigma_threshold,
)
print_qa_report(qa_issues)
print(f'\n{len(snr_excluded_planets)} planets excluded from SNR metrics')
print(f'{len(cos_incomplete_planets)} planets with incomplete COS exploration (STIS metrics kept)')


AU Mic b
  [model] case_level_count: obs config missing entirely: offset=1.0, grating='g130m', aperture='psa', lya='low_2sig'; obs config missing entirely: offset=1.0, grating='g130m', aperture='psa', lya='high_1sig'; obs config missing entirely: offset=1.0, grating='g130m', aperture='psa', lya='high_2sig'; obs config missing entirely: offset=1.0, grating='g130m', aperture='psa', lya='low_1sig'
    targets/au-mic/transit predictions/au-mic-b.outflow-tail-model.detection-sigmas.ecsv
  [model] case_combinations_missing: 3072 of 41472 combinations missing; examples: eta=0.31622776601683794, Tion=3.8336761161519592, sw_ram_pressure_at_pl=0.00043412911832764857, sw_velocity=400.00000000000006, mass=5.972000000000012e+27, time offset=1.0, grating='g130m', aperture='psa', lya reconstruction case='low_2sig'; eta=0.31622776601683794, Tion=0.3056532304787511, sw_ram_pressure_at_pl=2.015048867975775e-05, sw_velocity=400.00000000000006, mass=2.582019950108711e+28, time offset=1.0, grating='g130m'

In [9]:
print(qa_summary_table(qa_issues) if qa_issues else 'QA summary: no issues')

          check           count
------------------------- -----
         case_level_count    39
case_combinations_missing    39
          column_mismatch    17
             missing_file    18
              stale_mtime    33
          outdated_schema    15
             missing_meta     9
           mtime_mismatch     7
          incomplete_pair     4


In [11]:
##%% assemble table of properties

lya_bins = (-150, -50, 50, 150) * u.km/u.s

eval_rows = []
targets = sorted(targets)
for target in tqdm(targets):
    host = tutils.Host(target, host_catalog, planet_catalog)
    for planet in host.planets:
        row = {}

        row['hostname'] = host.hostname
        row['planet'] = planet.stela_suffix
        pkey = snr_planet_key(host, planet)
        row['snr qa\nflag'] = pkey in snr_excluded_planets
        row['COS snr\nqa flag'] = pkey in cos_incomplete_planets
        if row['snr qa\nflag']:
            row['snr qa\nreason'] = planet_qa_reason(qa_issues, pkey)
        elif row['COS snr\nqa flag']:
            row['snr qa\nreason'] = planet_cos_qa_reason(qa_issues, pkey)
        else:
            row['snr qa\nreason'] = ''

        if not row['snr qa\nflag']:
            try:
                # region model snrs
                snr_db = load_snr_db(planet, host, 'model')
                slctd_offsets, det_fracs, max_snrs = snr_db.offset_stats(
                    sigma_threshold, min_sample_check=5**4)

                row['best safe\ntransit offset'] = slctd_offsets[1]
                row['best overall\ntransit offset'] = slctd_offsets[2]

                off_lbls = ['no', 'safe', 'best']
                stat_sets = zip(det_fracs, max_snrs, off_lbls)
                for det_frac, max_snr, off_lbl in stat_sets:
                    row[f'sim {off_lbl} offset \nmax snr'] = max_snr
                    row[f'sim {off_lbl} offset \nfrac w snr > {sigma_threshold}'] = det_frac

                row['sim\nbest aperture'] = snr_db.meta['best base grating aperture']

                cos = snr_db.meta['COS considered']
                row['COS\nconsidered?'] = cos
                if cos and not row['COS snr\nqa flag']:
                    cos_snrs = snr_db.filter_obs_config(
                        grating='g130m', aperture='psa', offset='best safe')
                    cos_snrs = cos_snrs.clean_duplicates()
                    cosfrac, maxcossnr = cos_snrs.det_frac_and_max_snr(sigma_threshold)
                    row['sim COS safe offset\nmax snr'] = maxcossnr
                    row[f'sim COS safe offset\nfrac w snr > {sigma_threshold}'] = cosfrac
                    if det_fracs[1] > 0:
                        row['cos det\nfrac ratio'] = cosfrac / det_fracs[1]
                    row['cos snr\nratio'] = maxcossnr / max_snrs[1]
                elif cos and row['COS snr\nqa flag']:
                    for col in cos_metric_column_names(sigma_threshold):
                        row[col] = np.ma.masked

                row['H ionztn\ntime (h)'] = snr_db.geometric_mean_unique_tion_hours()

                # region flat transit
                snr_db_flat = load_snr_db(planet, host, 'flat')
                snr_db_flat = snr_db_flat.filter_obs_config(aperture='best', offset='best safe')
                snr_db_flat = snr_db_flat.clean_duplicates()
                flat_snr = snr_db_flat.median_case()['transit sigma']
                row['flat transit\nsnr'] = flat_snr
                row['flat transit\nbest aperture'] = snr_db_flat.meta['best base grating aperture']
            except Exception as exc:
                row['snr qa\nflag'] = True
                reason = f'compile_load_error: {exc}'
                if row['snr qa\nreason']:
                    row['snr qa\nreason'] = row['snr qa\nreason'] + '; ' + reason
                else:
                    row['snr qa\nreason'] = reason
                print(f'[{pkey}] SNR compile failed: {exc}')

        Flya = get_lya_flux(host)
        row['Lya Flux\n(erg s-1 cm-2)'] = Flya

        Mp = planet.params['pl_bmasse'].to_value('Mearth')
        Mp_err = 0.5 * (planet.params['pl_bmasseerr1'] - planet.params['pl_bmasseerr2'])
        Mp_prec = Mp_err / Mp
        if not np.isfinite(Mp_prec) or Mp_prec == 0:
            Mp_prec = np.ma.masked
        row['mass (Me)'] = Mp
        row['mass\nprecision'] = Mp_prec
        if planet.params['pl_bmassesrc'] == 'inferred from Rp':
            mass_source = 'M-R relationship'
        else:
            mass_source_rename = {'Mass': 'known', 'Msini': 'known', 'M-R relationship': 'M-R relationship'}
            mass_source = str(planet.params['pl_bmassprov'])
            mass_source = mass_source_rename[mass_source]
        row['mass\nsource'] = mass_source
        mass_flag = np.ma.masked
        if planet.params['pl_rade'] > 7 * u.Rearth:
            mass_flag = 'giant'
        if planet.params['flag_young']:
            mass_flag = 'young'
        row['mass\nflag'] = mass_flag

        row['radius (Re)'] = planet.params['pl_rade'].to_value('Rearth')
        row['orbital\nperiod (d)'] = planet.params['pl_orbper'].to_value('d')
        row['stellar\neff temp (K)'] = host.params['st_teff'].to_value('K')
        age = host.params['st_age'].to_value('Gyr')
        if not np.ma.is_masked(age):
            agelim_int = host.params['st_agelim']
            if not np.ma.is_masked(agelim_int):
                row['age\nlimit'] = catutils.limit_int2str[agelim_int]
            row['age (Gyr)'] = host.params['st_age'].to_value('Gyr')

        row['obsvtn\ngrating'] = host.anticipated_grating

        flag_cols = [name for name in planet_catalog.colnames if 'flag_' in name]
        for col in flag_cols:
            row[col] = planet.params[col]

        eval_rows.append(row)

# get column ordering from longest row
imax = np.argmax([len(row) for row in eval_rows])
ordered_cols = list(eval_rows[imax].keys())
for col in ('snr qa\nflag', 'snr qa\nreason'):
    if col in ordered_cols:
        ordered_cols.remove(col)
for i, col in enumerate(('snr qa\nflag', 'snr qa\nreason')):
    if col in eval_rows[imax]:
        ordered_cols.insert(2 + i, col)

eval_table = Table(rows=eval_rows)
eval_table = eval_table[ordered_cols]

eval_table['TIC'] = preloads.stela_names.loc['hostname', eval_table['hostname']]['tic_id']

# some grooming
for col in eval_table.colnames:
    if 'flag_' in col:
        eval_table[col] = eval_table[col].astype(bool)
formats_general = {
    'period': '.1f',
    'frac w': '.3f',
    'max snr': '.2f',
    'ratio': '.2f',
    'flux': '.1e',
    'radius': '.2f',
    'temp': '.0f',
    'ionztn': '.2f'
}
formats = {}
for substr, fmt in formats_general.items():
    for name in eval_table.colnames:
        if substr in name.lower():
            eval_table[name].format = fmt
            formats[name] = fmt




 56%|█████▋    | 70/124 [00:30<00:15,  3.51it/s]

[TOI-1468 c] SNR compile failed: [Errno 2] No such file or directory: '/Users/parke/Google Drive/Research/STELa/data/targets/toi-1468/transit predictions/toi-1468-c.simple-opaque-tail.detection-sigmas.ecsv'


 82%|████████▏ | 102/124 [00:45<00:07,  3.12it/s]

[TOI-5388 b] SNR compile failed: [Errno 2] No such file or directory: '/Users/parke/Google Drive/Research/STELa/data/targets/toi-5388/transit predictions/toi-5388-b.simple-opaque-tail.detection-sigmas.ecsv'


100%|██████████| 124/124 [00:55<00:00,  2.25it/s]


In [12]:
##%% match in the catalog of escape detections

escape_detections = catutils.escape_catalog_merge_targets('download')

det_ids = np.char.add(escape_detections['Target Star'], escape_detections['Planet Letter'])
pcat_ids = np.char.add(planet_catalog['hostname'].astype(str),
                       dbutils.planet_suffixes(planet_catalog).astype(str))
eval_ids = np.char.add(eval_table['hostname'], eval_table['planet'])

# check to be sure names in escape detections match into planet catalog
suspect = ~np.isin(det_ids, pcat_ids)
if np.any(suspect):
    print("These names don't have a match in the planet catalog."
          "Note this could be do to cuts to planet catalog, such as requiring < 100 d periods (e.g., HD 136352d)."
          "Or because they aren't in the planet catalog your using to build the table (e.g. HAT-P-26 is not in any evals).")
    print(det_ids[suspect])

# match using name + letter
det_colnames = [name for name in escape_detections.colnames if 'detected' in name.lower()]
det_slim = escape_detections[det_colnames]
det_slim['temp'] = det_ids
eval_table['temp'] = eval_ids
eval_table = table.join(eval_table, det_slim, keys='temp', join_type='left')
eval_table.remove_column('temp')




These names don't have a match in the planet catalog.(Note this could be do to cuts to planet catalog, such as requiring < 100 d periods (e.g., HD 136352d).)
['HAT-P-26b' 'HIP 67522b' 'LTT 9779b' 'TOI-1268b' 'TOI-1728b' 'TOI-2136b'
 'WASP-127b' 'WASP-52b' 'WASP-77 Ab']


In [14]:
##%% match in requested targets

files = list(paths.stage2_requests.glob('*.txt'))
path_check_table, = paths.selection_intermediates.glob('*pt1*.ecsv')
check_table = catutils.load_and_mask_ecsv(path_check_table)

def any_tois(planet_names):
    for name in planet_names:
        x = re.findall(r'TOI-\d+\.0\d', name)
        if x:
            return True
    return False

eval_table['TICletter'] = np.char.add( # temporary column for matchin
    eval_table['TIC'].astype(str),
    eval_table['planet']
)
check_table['TICletter'] = np.char.add( # temporary column for matching
    check_table['tic_id'].astype(str),
    check_table['pl_letter'].astype(str).filled('')
)
for file in files:
    requested_planets = catutils.read_requested_targets(file)
    if any_tois(requested_planets):
        raise NotImplementedError

    # match with TIC to ensure avoid misses
    hosts_letters = list(map(dbutils.split_hostname_planet_letter, requested_planets))
    hosts, letters = zip(*hosts_letters)
    tics = dbutils.query.query_simbad_for_tic_ids(hosts)
    tics_letters = np.char.add(tics, letters)

    # mark matches in a "requested" column
    request_name = 'requested\n' + requested_planets.name
    eval_table[request_name] = np.isin(eval_table['TICletter'], tics_letters)

    # print requested targets with no match in STELa tables
    not_in_list = ~np.isin(tics_letters, check_table['TICletter'])
    if np.any(not_in_list):
        print(f'\n{requested_planets.name} planets not matched to any planet known to STELa:')
        for name in requested_planets[not_in_list]:
            print(f'\n\t{name}')
        print('Consider checking that they are correctly named in the requested list txt file.')

eval_table.remove_column('TICletter')




In [15]:
##%% save table

# save csv to open in spreadsheet viewers
eval_filename = 'stage2_evaluation_metrics.csv'
eval_path = paths.catalogs / eval_filename
eval_table.write(eval_path, overwrite=True, formats=formats, fast_writer=False)

# save as an ecsv too for round tripping
eval_table_ecsv = eval_table.copy()
for name in eval_table_ecsv.colnames:
    eval_table_ecsv.rename_column(name, name.replace('\n', ' '))
eval_path_ecsv = paths.catalogs / eval_filename.replace('csv', 'ecsv')
eval_table_ecsv.write(eval_path_ecsv, overwrite=True)
